## Task 2 로직 테스트

- 1. 질문 키워드 분석 (조건들을 세부적으로 나누기)
- 2. yfinance에서 위 키워드를 바탕으로 어떤 값들을 가져와야 되는지 판단
- 3. 질문에 맞게 값 계산
- 4. 응답

In [1]:
import os
import dotenv
from langchain_naver import ChatClovaX

dotenv.load_dotenv()

chat = ChatClovaX(
    api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
    model="HCX-005",
    temperature=0.2,
    max_tokens=None,
    timeout=None,
    # max_retries,
    # repeat_penalt,
    # repetition_penalty,
    # stop,
    # seed,
    # top_k,
    # top_p,
)

In [2]:
prompt = """
# ROLE: system
당신은 주식 조건검색 문장을 구조화된 JSON 으로 변환하는 분석 전문가입니다.
다른 말은 절대 출력하지 말고, 아래 "반드시 지켜야 하는 스키마"에 100 % 맞는 JSON 만 출력하세요.

<date_range 핵심 규칙 - 매우 중요!>
date_range는 "해당 조건을 계산하기 위해 최소 며칠의 데이터가 필요한가?"를 의미합니다.

**절대 규칙:**
- change_rate (등락률) = 2 (당일 종가 + 전일 종가 필요)
- volume_change_rate (거래량 변화율) = 2 (당일 거래량 + 전일 거래량 필요)  
- volume_abs (거래량 절대값) = 1 (당일 거래량만 필요)
- price (가격) = 1 (당일 종가만 필요)

**절대 예외 없음!**

<date_range 예시>
- "등락률 5% 이상" → change_rate → date_range = 2
- "거래량이 전날대비 100% 증가" → volume_change_rate → date_range = 2
- "거래량이 100만주 이상" → volume_abs → date_range = 1
- "종가가 1만원 이상" → price → date_range = 1

<스키마>
{{
  "type":           string,   # ↓ 표에서 하나만
  "date":           "YYYY-MM-DD",
  "market":         "KOSPI" | "KOSDAQ" | "ALL",
  "conditions": [
    {
      "field":      "change_rate" | "volume_change_rate" | "volume_abs" | "price",
      "operator":   ">" | ">=" | "<" | "<=" | "range",
      "date_range": number,         
      "value":      number | null,
      "min_value":  number | null,
      "max_value":  number | null,
      "unit":       "percent" | "shares" | "won",
      "description": string
    }
  ],
  "keywords":   [string, ...],      # 최대 5개, 핵심 명사·숫자 위주
  "complexity": "simple" | "complex",
  "confidence": number,             # 0.0-1.0
  "reasoning":  string              # 30자 이하 요약
}}

<type 분류 표>
| 문장 특징                                    | type 값                    |
|----------------------------------------------|----------------------------|
| “KOSPI 시장 … 등락률 …”                      | KOSPI_등락률               |
| “KOSPI 시장 … 종가(=가격) …”                 | KOSPI_가격범위             |
| “KOSDAQ 시장 … 거래량 전날대비 …”            | KOSDAQ_거래량변화          |
| “KOSDAQ 시장 … 거래량 …주 이상”              | KOSDAQ_거래량              |
| “KOSDAQ 시장 … 종가 …원 이상/이하”           | KOSDAQ_가격범위            |
| “등락률 + 거래량변화” 같은 다중 조건 포함     | 복합조건                   |
| 시장 지정 없고 “등락률 …”만                  | 등락률                     |
| 시장 지정 없고 “거래량 …주 이상/이하”        | 거래량절대값               |
| 시장 지정 없고 “거래량 전날대비 …% 이상/이하”| 거래량변화율               |
| 시장 지정 없고 “종가 …원 이상/이하”          | 가격범위                   |

<필드 결정 규칙>
- “등락률” → field == change_rate, unit == percent
- “거래량이 전날대비” → field == volume_change_rate, unit == percent
- “거래량이 100만주” → field == volume_abs, unit == shares (숫자는 ‘만’, ‘천’ 단위 환산 하지 말고 실제 주식 수로 변환: 100만주 → 1000000)
- “종가가 …원” → field == price, unit == won (만원·천원·백원 단위도 숫자로 환산: 1만원 → 10000)

<연산자 규칙>
- “~이상” ⇒ operator == ">="
- “~초과” ⇒ ">"
- “~이하” ⇒ "<="
- “~미만” ⇒ "<"
- “A 이상 B 이하 범위” ⇒ operator == "range", min_value == A, max_value == B

<복잡도 판단>
- 조건 1개 ⇒ simple ; 2개 이상 ⇒ complex 

<출력 예시>
{{
  "type": "복합조건",
  "date": "2025-04-02",
  "market": "ALL",
  "conditions": [
    {
      "field": "change_rate",
      "operator": ">=",
      "date_range": 2,                       # 전일 종가 필요
      "value": 3,
      "unit": "percent",
      "min_value": null,
      "max_value": null,
      "description": "등락률 +3% 이상"
    },
    {
      "field": "volume_change_rate",
      "operator": ">=",
      "date_range": 2,                       # 전일 거래량 필요
      "value": 100,
      "unit": "percent",
      "min_value": null,
      "max_value": null,
      "description": "거래량 전날대비 100% 이상 증가"
    }
  ],
  "keywords": ["2025-04-02", "등락률", "거래량", "3", "100"],
  "complexity": "complex",
  "confidence": 0.93,
  "reasoning": "두 조건 모두 전일 비교"
}}

"""

text = '2025-06-25에 KOSPI 시장에서 등락률이 +10% 이상인 종목을 모두 보여줘'

messages = [("system", prompt),
            ("human", text)]

ai_msg = chat.invoke(messages)
ai_msg.content

'```json\n{\n  "type": "KOSPI_등락률",\n  "date": "2025-06-25",\n  "market": "KOSPI",\n  "conditions": [\n    {\n      "field": "change_rate",\n      "operator": ">=",\n      "date_range": 2,\n      "value": 10,\n      "unit": "percent",\n      "min_value": null,\n      "max_value": null,\n      "description": "등락률 +10% 이상"\n    }\n  ],\n  "keywords": ["2025-06-25", "KOSPI", "등락률", "10%"],\n  "complexity": "simple",\n  "confidence": 0.95,\n  "reasoning": "단순한 등락률 조건 검색"\n}\n```'

In [3]:
def parse_llm_json(text: str) -> dict:
    import json, re, ast

    # 1)  {...} 블록만 추출
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise ValueError("JSON 블록을 찾을 수 없습니다.")
    j = m.group()

    # 2) JSON → dict
    try:
        return json.loads(j)
    except json.JSONDecodeError:
        return ast.literal_eval(j)   # fallback

parse_llm_json(ai_msg.content)

{'type': 'KOSPI_등락률',
 'date': '2025-06-25',
 'market': 'KOSPI',
 'conditions': [{'field': 'change_rate',
   'operator': '>=',
   'date_range': 2,
   'value': 10,
   'unit': 'percent',
   'min_value': None,
   'max_value': None,
   'description': '등락률 +10% 이상'}],
 'keywords': ['2025-06-25', 'KOSPI', '등락률', '10%'],
 'complexity': 'simple',
 'confidence': 0.95,
 'reasoning': '단순한 등락률 조건 검색'}

In [13]:
import pandas as pd

llm_question_analysis = parse_llm_json(ai_msg.content)
market_list = pd.read_csv("/Users/yoon/MIRAE_ASSET_AI-Festival/krx_tickers_cleaned.csv")

def get_tickers(answer):
    if answer['market'] == "KOSPI":
        ticker_lists = market_list.loc[market_list["market"] == "KOSPI", "code"].tolist()
        # .KS 접미사 추가
        ticker_lists = [f"{ticker}.KS" for ticker in ticker_lists]
        print(len(ticker_lists))
        return ticker_lists
    elif answer['market'] == "KOSDAQ":
        ticker_lists = market_list.loc[market_list["market"] == "KOSDAQ", "code"].tolist()
        # .KQ 접미사 추가
        ticker_lists = [f"{ticker}.KQ" for ticker in ticker_lists]
        print(len(ticker_lists))
        return ticker_lists
    else:
        # 전체 시장인 경우 각각에 맞는 접미사 추가
        kospi_tickers = market_list.loc[market_list["market"] == "KOSPI", "code"].tolist()
        kosdaq_tickers = market_list.loc[market_list["market"] == "KOSDAQ", "code"].tolist()
        
        ticker_lists = [f"{ticker}.KS" for ticker in kospi_tickers] + \
                      [f"{ticker}.KQ" for ticker in kosdaq_tickers]
        print(len(ticker_lists))
        return ticker_lists

target_market = get_tickers(llm_question_analysis)

962


In [14]:
import yfinance as yf
from typing import List
from datetime import datetime, timedelta
from typing import Dict, Callable, List, Set, Tuple
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# 1. 유틸: 연산자 함수 테이블
OPS: Dict[str, Callable] = {
    ">":  lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<":  lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
    "range": lambda a, min_val, max_val: min_val <= a <= max_val
}

# 2. 유틸: yfinance 에서 필요한 칼럼
FIELD_COLS = {
    "change_rate":        "Close",
    "volume_change_rate": "Volume",
    "volume_abs":         "Volume",
    "price":              "Close",
}

# 3. 전일 대비 지표 계산
def compute_metrics(df: pd.DataFrame, field: str) -> float:
    """
    한 종목의 가격·거래량 DataFrame → 지표 값(전일 대비 %, 절댓값 등)
    df.index: DatetimeIndex, 오름차순.
    """
    today = df.iloc[-1]
    prev  = df.iloc[-2] if len(df) >= 2 else None   # 전일이 없으면 None

    if field == "change_rate":
        if prev is None or prev["Close"] == 0:
            return float("nan")
        return (today["Close"] / prev["Close"] - 1) * 100

    if field == "volume_change_rate":
        if prev is None or prev["Volume"] == 0:
            return float("nan")
        return (today["Volume"] / prev["Volume"] - 1) * 100

    if field == "volume_abs":
        return today["Volume"]

    if field == "price":
        return today["Close"]

    raise ValueError(f"지원하지 않는 field: {field}")

In [20]:
import time
def download_in_batches(tickers, start_dt, end_dt, batch_size=100):
    """배치 단위로 다운로드 후 멀티인덱스 DataFrame으로 합치기"""
    all_ticker_data = {}
    total_batches = len(tickers) // batch_size + (1 if len(tickers) % batch_size > 0 else 0)
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} tickers)")
        
        try:
            # 배치 단위로 다운로드
            hist = yf.download(
                tickers=" ".join(batch),
                start=start_dt,
                end=end_dt,
                interval="1d",
                group_by="ticker",
                progress=False,
                threads=True,
                auto_adjust=False,
            )
            
            if not hist.empty:
                # 단일 종목인 경우 멀티인덱스로 변환
                if len(batch) == 1:
                    ticker = batch[0]
                    hist.columns = pd.MultiIndex.from_product([[ticker], hist.columns])
                
                # 각 ticker별로 데이터 저장
                for ticker in batch:
                    try:
                        if len(batch) == 1:
                            ticker_data = hist[ticker]
                        else:
                            ticker_data = hist[ticker]
                        
                        if not ticker_data.empty and not ticker_data.isna().all().all():
                            all_ticker_data[ticker] = ticker_data
                    except (KeyError, AttributeError):
                        print(f"No data for {ticker}")
                
                print(f"✓ Batch {batch_num} completed")
            else:
                print(f"✗ Batch {batch_num} returned empty data")
                
        except Exception as e:
            print(f"✗ Batch {batch_num} failed: {e}")
            # 실패한 배치는 개별 처리
            for ticker in batch:
                try:
                    single_hist = yf.download(ticker, start=start_dt, end=end_dt, progress=False)
                    if not single_hist.empty:
                        all_ticker_data[ticker] = single_hist
                        print(f"Individual download success: {ticker}")
                except:
                    print(f"Failed to download {ticker}")
        
        time.sleep(1)
    
    # 모든 ticker 데이터를 멀티인덱스 DataFrame으로 합치기
    if all_ticker_data:
        print(f"Combining data for {len(all_ticker_data)} tickers...")
        
        # 멀티인덱스 DataFrame 생성
        combined_data = pd.concat(all_ticker_data, axis=1)
        
        # 컬럼 순서 정리 (Price 기준으로)
        combined_data.columns.names = ['Ticker', 'Price']
        
        print(f"Final data shape: {combined_data.shape}")
        print(f"Columns: {combined_data.columns.get_level_values(1).unique().tolist()}")
        
        return combined_data
    else:
        print("No data downloaded")
        return pd.DataFrame()

In [26]:
def screen_stocks(q: dict) -> List[str]:
    """
    q : LLM JSON 파싱 결과.
    반환 : 모든 조건을 동시에 만족하는 종목 코드 리스트(중복 제거·정렬).
    """
    tickers  = get_tickers(q)
    if not tickers:
        return []

    # 거래량 / 종가 계산 컬럼 가져오기
    need_col = FIELD_COLS[q['conditions'][0]['field']]
    print(need_col)

    # 여러 조건 중 가장 긴 date_range 를 뽑아 한 번만 다운로드
    max_range = max(c["date_range"] for c in q["conditions"])
    print(max_range)

    # 질문에 있는 날짜
    tgt_date  = datetime.strptime(q["date"], "%Y-%m-%d")
    print(tgt_date)

    # 주말·휴일 여유
    start_dt  = tgt_date - timedelta(days=5)            
    print(start_dt)

    # 끝 날짜
    end_dt    = tgt_date + timedelta(days=1)          
    print(end_dt)

    # 배치 다운로드
    data = download_in_batches(tickers, start_dt, end_dt, batch_size=100)

    # 거래량 / 종가 계산 컬럼만 있는 데이터프레임
    only_need_col = data.xs(need_col, level=1, axis=1)
    
    print(only_need_col.head(10))

    # 전날 대비 거래량 / 등락률 계산
    if q['conditions'][0]['date_range'] == 2:
        only_need_col_edit = only_need_col.pct_change() * 100
        final_data = only_need_col_edit.iloc[-1]
        print("계산---------------------------")
        print(final_data.head(10))
    else:
        final_data = only_need_col.iloc[-1]
        print("계산---------------------------")
        print(final_data.head(10))

    # 연산자에 따른 계산 다르게
    def apply_operator(df, operator, value, min_value=None, max_value=None):
        """
        데이터프레임에 operator 조건을 적용하는 함수
        """
        if operator == "range":
            return (df >= min_value) & (df <= max_value)
        else:
            return OPS[operator](df, value)
        
    result = apply_operator(final_data, q['conditions'][0]['operator'], q['conditions'][0]['value'], q['conditions'][0]['min_value'], q['conditions'][0]['max_value'])

    # 조건 만족해서 True인 행의 Ticker들
    true_tickers = result[result == True].index.tolist()
    print("조건을 만족하는 ticker들:", true_tickers)

    # Ticker들을 다시 종목명으로 변환
    stock_names = {}
    stock_info = pd.read_csv('/Users/yoon/MIRAE_ASSET_AI-Festival/krx_tickers_cleaned.csv')  
    ticker_to_name = dict(zip(stock_info['code'], stock_info['name']))
    for ticker in true_tickers:
        clean_ticker = ticker.replace('.KS', '').replace('.KQ', '')
        stock_name = ticker_to_name.get(clean_ticker, ticker)
        stock_names[ticker] = stock_name

    return stock_names

In [27]:
screen_stocks(llm_question_analysis)

962
Close
2
2025-06-25 00:00:00
2025-06-20 00:00:00
2025-06-26 00:00:00
Processing batch 1/10 (100 tickers)
✓ Batch 1 completed
Processing batch 2/10 (100 tickers)
✓ Batch 2 completed
Processing batch 3/10 (100 tickers)
✓ Batch 3 completed
Processing batch 4/10 (100 tickers)
✓ Batch 4 completed
Processing batch 5/10 (100 tickers)
✓ Batch 5 completed
Processing batch 6/10 (100 tickers)
✓ Batch 6 completed
Processing batch 7/10 (100 tickers)
✓ Batch 7 completed
Processing batch 8/10 (100 tickers)
✓ Batch 8 completed
Processing batch 9/10 (100 tickers)
✓ Batch 9 completed
Processing batch 10/10 (62 tickers)
✓ Batch 10 completed
Combining data for 962 tickers...
Final data shape: (4, 5772)
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
Ticker      095570.KS  006840.KS  027410.KS  282330.KS  138930.KS  001460.KS  \
Date                                                                           
2025-06-20     3985.0    11850.0     4285.0   125000.0    11790.0    37750.0   


{'375500.KS': 'DL이앤씨',
 '37550L.KS': 'DL이앤씨2우(전환)',
 '009070.KS': 'KCTC',
 '004140.KS': '동방',
 '000400.KS': '롯데손해보험',
 '001430.KS': '세아베스틸지주',
 '006220.KS': '제주은행',
 '005690.KS': '파미셀',
 '015760.KS': '한국전력'}